In [6]:
import hashlib
import datetime

def sha256(data: str) -> str:
    """Returns the SHA-256 hash of the given string."""
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

def build_merkle_tree(transactions: list[str]) -> str:
    """Builds the Merkle tree and returns the Merkle root using sorted-pair hashing."""
    if not transactions:
        return ""
    leaves = [sha256(tx) for tx in transactions]
    while len(leaves) > 1:
        new_leaves = []
        for i in range(0, len(leaves), 2):
            left = leaves[i]
            right = leaves[i + 1] if i + 1 < len(leaves) else leaves[i]
            if left > right:
                left, right = right, left
            new_leaves.append(sha256(left + right))
        leaves = new_leaves
    return leaves[0]

def merkle_proof(transactions: list[str], target_tx: str) -> list[str]:
    """Generate a Merkle proof using sorted-pair hashing."""
    target_hash = sha256(target_tx)
    leaves = [sha256(tx) for tx in transactions]
    
    if target_hash not in leaves:
        raise ValueError("Target transaction not found in transactions.")
        
    proof = []
    index = leaves.index(target_hash)
    
    while len(leaves) > 1:
        if index % 2 == 0:
            sibling = leaves[index + 1] if index + 1 < len(leaves) else leaves[index]
        else:
            sibling = leaves[index - 1]
        proof.append(sibling)
        
        new_leaves = []
        for i in range(0, len(leaves), 2):
            left = leaves[i]
            right = leaves[i + 1] if i + 1 < len(leaves) else leaves[i]
            if left > right:
                left, right = right, left
            new_leaves.append(sha256(left + right))
                
        leaves = new_leaves
        index //= 2
        
    return proof

In [7]:
transactions = ["tx1", "tx2", "tx3", "tx4"]
root = build_merkle_tree(transactions)
print(f"Merkle Root: {root}")

Merkle Root: ecca613e40569e192915dc30585d9a77173297473be83c5999a7033571f57326


In [8]:
def verify_merkle_proof(proof: list[str], target_tx: str, root: str) -> bool:
    """Verify a Merkle proof using sorted-pair hashing."""
    computed_hash = sha256(target_tx)
    for sibling in proof:
        if computed_hash < sibling:
            computed_hash = sha256(computed_hash + sibling)
        else:
            computed_hash = sha256(sibling + computed_hash)
    return computed_hash == root

In [9]:
import hashlib
import datetime

class Block:
    def __init__(self, index, timestamp, transactions, previous_hash):
        self.index = index
        self.timestamp = timestamp
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.merkle_root = build_merkle_tree(transactions)
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = f"{self.index}{self.timestamp}{self.merkle_root}{self.previous_hash}"
        return sha256(block_string)

class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(0, str(datetime.datetime.now()), ["Genesis Block"], "0")

    def add_block(self, transactions):
        last_block = self.chain[-1]
        new_block = Block(len(self.chain), str(datetime.datetime.now()), transactions, last_block.hash)
        self.chain.append(new_block)



In [10]:
# --- Execution & Verification ---
transactions = ["tx1", "tx2", "tx3", "tx4"]
root = build_merkle_tree(transactions)
proof = merkle_proof(transactions, "tx3")

print(f"Merkle Root: {root}")
print(f"Merkle Proof for 'tx3': {proof}")
print(f"Verification: {verify_merkle_proof(proof, 'tx3', root)}")  # Outputs: True

blockchain = Blockchain()
blockchain.add_block(["tx1", "tx2", "tx3"])
blockchain.add_block(["tx4", "tx5"])

for block in blockchain.chain:
    print(f"Block #{block.index}: {block.hash}")
    print(f"Merkle Root: {block.merkle_root}")
    print(f"Previous Hash: {block.previous_hash}\n")

Merkle Root: ecca613e40569e192915dc30585d9a77173297473be83c5999a7033571f57326
Merkle Proof for 'tx3': ['41b637cfd9eb3e2f60f734f9ca44e5c1559c6f481d49d6ed6891f3e9a086ac78', '472b125cd7be33e50ea6ef56dd83389f4ab7a5e1907c082d0d5bfa86431c953a']
Verification: True
Block #0: 18b1083b9af6684adf7c7ea4d03a9a0dbcb64db787463748f29dd64cbca13821
Merkle Root: 89eb0ac031a63d2421cd05a2fbe41f3ea35f5c3712ca839cbf6b85c4ee07b7a3
Previous Hash: 0

Block #1: 9659123c0ba8a5b3ca0825c2782f2cc1c007ddfc931a1e9ff21b261adc112c0d
Merkle Root: daebb2f1c68ae4c20be2de6f13d987d195a5e9f037fb277d0031d488268b155a
Previous Hash: 18b1083b9af6684adf7c7ea4d03a9a0dbcb64db787463748f29dd64cbca13821

Block #2: bd4b34bdda2d87419dda477d75bf6122257d98532d211a03a947e11983d733b2
Merkle Root: 8fca72ca8ee89e32a7483de0f66ee262e01012126cee65563fdee5ac36b8903c
Previous Hash: 9659123c0ba8a5b3ca0825c2782f2cc1c007ddfc931a1e9ff21b261adc112c0d

